# 0. Overview

This notebook analyzes whether the ENSO-rainfall relationship over the Maritime Continent changed between **Old (1981-2006)** and **New (2007-2020)** periods, compared against the **Full (1981-2020)** baseline.

Scientific objective:
- Separate mean-state change (climatology) from teleconnection change (correlation/regression vs Niño3.4) and event-conditioned responses (El Niño / La Niña composites).

Period definitions:
- Full: 1981-2020
- Old: 1981-2006
- New: 2007-2020
- Delta: New minus Old

Variables and diagnostics:
- Rainfall (MSWEP)
- Diagnostics for each family: climatology, teleconnection, ENSO composites

# 1. Import Library

In [ ]:
from pathlib import Path

# Define data paths (relative to notebook location)
RAINFALL_PATH = Path('../external/ClimateData/mswep-monthly/mswep_monthly_combined.nc').resolve()
WIND_PATH = Path('../external/ClimateData/era5-monthly/era5monthly_uvq_1980-2020.nc').resolve()
MFC_PATH = Path('../external/ClimateData/era5-monthly/vimf.nc').resolve()
SVP_PATH = Path('analysis_26-14_rev2/psi_chi_windparts_850.nc').resolve()
NINO34_PATH = Path('../external/ClimateData/index-monthly/nino34.anom.csv').resolve()

import numpy as np
import xarray as xr
import pandas as pd
import dask
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.pyplot as plt

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica']
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Helvetica'

# 2. Open Data & Pre-Process

#### Open MSWEP

In [ ]:
mswep_path = rainfall_path = RAINFALL_PATH
mswep_chunks = {'time': 12}

ds_mswep = xr.open_dataset(mswep_path, chunks=mswep_chunks)['precipitation']
ds_mswep = ds_mswep.assign_coords(lon=(ds_mswep.lon % 360)).sortby('lon')
#slice wilayah maritime continent samudra pasifik dan samudra hindia
ds_mswep = ds_mswep.sel(
    time=slice('1980-12-01', '2020-02-29'),
    lat=slice(21, -21),
    lon=slice(70, 290),
)
# keep DJF only, with December assigned to the following DJF year
month_mask = ds_mswep.time.dt.month.isin([12, 1, 2])
djf_year = xr.where(ds_mswep.time.dt.month == 12, ds_mswep.time.dt.year + 1, ds_mswep.time.dt.year)
ds_mswep = ds_mswep.sel(time=month_mask).assign_coords(djf_year=('time', djf_year.sel(time=month_mask).data))


#### Open NINO34 index

In [ ]:
nino34_path = nino34_path = NINO34_PATH
nino34_column = '   Nino Anom 3.4 Index  using ersstv5 from CPC  missing value -99.99 https://psl.noaa.gov/data/timeseries/month/'

df_nino34 = pd.read_csv(
    nino34_path,
    usecols=['Date', nino34_column],
    parse_dates=['Date'],
)
df_nino34 = df_nino34.set_index('Date').loc['1980-12-01':'2020-02-29']
df_nino34 = df_nino34[df_nino34.index.month.isin([12, 1, 2])].copy()
df_nino34['djf_year'] = df_nino34.index.year + (df_nino34.index.month == 12).astype('int8')
df_nino34['DJF'] = 'DJF' + df_nino34['djf_year'].astype(str)


#### Define Period

In [ ]:
full_years = np.arange(1981, 2021)
past_years = np.arange(1981, 2007)
recent_years = np.arange(2007, 2021)

rain_clim = ds_mswep.sel(time=ds_mswep.djf_year.isin(full_years))
rain_past = rain_clim.sel(time=rain_clim.djf_year.isin(past_years))
rain_recent = rain_clim.sel(time=rain_clim.djf_year.isin(recent_years))

nino34_clim = df_nino34[df_nino34['djf_year'].isin(full_years)].copy()
nino34_past = nino34_clim[nino34_clim['djf_year'].isin(past_years)].copy()
nino34_recent = nino34_clim[nino34_clim['djf_year'].isin(recent_years)].copy()

period_coord = xr.DataArray(
    np.where(rain_clim.djf_year <= 2006, 'past', 'recent'),
    coords={'time': rain_clim.time},
    dims='time',
    name='period',
)
rain_period = rain_clim.assign_coords(period=period_coord)


#### Define Area

In [ ]:
# Define the extent and ticks 
# ip = Indonesia-Pacific region
ip_extent = [70, 290, -21, 21]
ip_yticks = [-15, 0, 15]
ip_xticks = np.arange(90, 291, 30)

In [ ]:
# Define the extent and ticks 
# mc: maritime continent
mc_extent = [90, 155, -15, 20]  # [lon_min, lon_max, lat_min, lat_max]
mc_yticks = np.arange(-10, 11, 10)
mc_xticks = np.arange(100, 151, 10)

# 3. Plot Climatology

In [ ]:
# Compute the shared climatology means once and reuse the period grouping
full_plot, period_means = dask.compute(
    rain_clim.mean('time'),
    rain_period.groupby('period').mean('time'),
)

# Transpose after loading (faster on in-memory numpy arrays)
full_plot = full_plot.transpose('lat', 'lon')
past_plot = period_means.sel(period='past').transpose('lat', 'lon')
recent_plot = period_means.sel(period='recent').transpose('lat', 'lon')
diff_plot = recent_plot - past_plot


### Indo Pasifik

In [ ]:
plots = [
    (full_plot, '(a) DJF 1981-2020', 'YlGnBu', np.arange(0, 501, 20), np.arange(0, 501, 100), 'Curah hujan (mm/bulan)', 'max'),
    (past_plot, '(b) DJF 1981-2006', 'YlGnBu', np.arange(0, 501, 20), np.arange(0, 501, 100), 'Curah hujan (mm/bulan)', 'max'),
    (recent_plot, '(c) DJF 2007-2020', 'YlGnBu', np.arange(0, 501, 20), np.arange(0, 501, 100), 'Curah hujan (mm/bulan)', 'max'),
    (diff_plot, '(d) Selisih (c)-(b)', 'PuOr', np.arange(-50, 51, 5), np.arange(-50, 51, 25), 'Selisih (mm/bulan)', 'both'),
]

for data, title, cmap, levels, ticks, cbar_label, extend in plots:
    fig = plt.figure(figsize=(12, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    img = data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        add_labels=False,
        infer_intervals=True,
    )

    ax.coastlines(resolution='10m', linewidth=0.5, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(ip_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(ip_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(ip_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=16)
    ax.set_title(title, fontsize=18, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.025])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=16, labelpad=10)
    cbar.ax.tick_params(labelsize=16)

    plt.show()


## Maritime Continent

In [ ]:
plots = [
    (full_plot, '(a) DJF 1981-2020', 'YlGnBu', np.arange(0, 501, 20), np.arange(0, 501, 100), 'Curah hujan (mm/bulan)', 'max'),
    (past_plot, '(b) DJF 1981-2006', 'YlGnBu', np.arange(0, 501, 20), np.arange(0, 501, 100), 'Curah hujan (mm/bulan)', 'max'),
    (recent_plot, '(c) DJF 2007-2020', 'YlGnBu', np.arange(0, 501, 20), np.arange(0, 501, 100), 'Curah hujan (mm/bulan)', 'max'),
    (diff_plot, '(d) Selisih (c)-(b)', 'PuOr', np.arange(-50, 51, 5), np.arange(-50, 51, 25), 'Selisih (mm/bulan)', 'both'),
]

for data, title, cmap, levels, ticks, cbar_label, extend in plots:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    img = data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        add_labels=False,
        infer_intervals=True,
    )

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(np.arange(100, 151, 20), crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


# 4. Plot Composite

### A. Analisis El Nino

In [ ]:
# define el nino years based on DJF-mean nino34 index threshold
elnino_threshold = 0.5
nino34_clim_djf = nino34_clim.groupby('djf_year')[nino34_column].mean().reset_index()
nino34_past_djf = nino34_past.groupby('djf_year')[nino34_column].mean().reset_index()
nino34_recent_djf = nino34_recent.groupby('djf_year')[nino34_column].mean().reset_index()

nino34_clim_elnino = nino34_clim_djf[nino34_clim_djf[nino34_column] >= elnino_threshold]
nino34_past_elnino = nino34_past_djf[nino34_past_djf[nino34_column] >= elnino_threshold]
nino34_recent_elnino = nino34_recent_djf[nino34_recent_djf[nino34_column] >= elnino_threshold]

elnino_years_clim = sorted(nino34_clim_elnino['djf_year'].tolist())
elnino_years_past = sorted(nino34_past_elnino['djf_year'].tolist())
elnino_years_recent = sorted(nino34_recent_elnino['djf_year'].tolist())
print('El Nino DJF years (1981-2020):', elnino_years_clim)
print('El Nino DJF years (1981-2006):', elnino_years_past)
print('El Nino DJF years (2007-2020):', elnino_years_recent)

rain_clim_elnino = rain_clim.sel(time=rain_clim.djf_year.isin(elnino_years_clim))
rain_past_elnino = rain_past.sel(time=rain_past.djf_year.isin(elnino_years_past))
rain_recent_elnino = rain_recent.sel(time=rain_recent.djf_year.isin(elnino_years_recent))

rain_clim_elnino, rain_past_elnino, rain_recent_elnino = dask.compute(
    rain_clim_elnino.mean('time'),
    rain_past_elnino.mean('time'),
    rain_recent_elnino.mean('time'),
)

rain_clim_elnino = rain_clim_elnino.transpose('lat', 'lon')
rain_past_elnino = rain_past_elnino.transpose('lat', 'lon')
rain_recent_elnino = rain_recent_elnino.transpose('lat', 'lon')

rain_clim_elnino_anom = rain_clim_elnino - full_plot
rain_past_elnino_anom = rain_past_elnino - past_plot
rain_recent_elnino_anom = rain_recent_elnino - recent_plot

rain_clim_elnino_anom = rain_clim_elnino_anom.reset_coords(drop=True)
rain_past_elnino_anom = rain_past_elnino_anom.reset_coords(drop=True)
rain_recent_elnino_anom = rain_recent_elnino_anom.reset_coords(drop=True)


#### plot komposit indo pasifik

In [ ]:
#data, title, cmap, levels, ticks, cbar_label, extend
plots = [
    (rain_clim_elnino_anom, '(a) Komposit El Nino DJF (1981 - 2020)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    (rain_past_elnino_anom, '(b) Komposit El Nino DJF (1981 - 2006)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    (rain_recent_elnino_anom, '(c) Komposit El Nino DJF (2007 - 2020)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    ((rain_recent_elnino_anom - rain_past_elnino_anom), '(d) Selisih (c) - (b)', 'BrBG', 'Selisih (mm/bulan)', np.arange(-100, 101, 5), np.arange(-100, 101, 25), 'both'),
]

In [ ]:

for data, title, cmap, cbar_label, levels, ticks, extend in plots:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(ip_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(ip_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(ip_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


#### plot mc

In [ ]:
for data, title, cmap, cbar_label, levels, ticks, extend in plots:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


### B. Analisis La Nina

In [ ]:
# define la nina years based on DJF-mean nino34 index threshold
lanina_threshold = -0.5

nino34_clim_lanina = nino34_clim_djf[nino34_clim_djf[nino34_column] <= lanina_threshold]
nino34_past_lanina = nino34_past_djf[nino34_past_djf[nino34_column] <= lanina_threshold]
nino34_recent_lanina = nino34_recent_djf[nino34_recent_djf[nino34_column] <= lanina_threshold]

lanina_years_clim = sorted(nino34_clim_lanina['djf_year'].tolist())
lanina_years_past = sorted(nino34_past_lanina['djf_year'].tolist())
lanina_years_recent = sorted(nino34_recent_lanina['djf_year'].tolist())
print('La Nina DJF years (1981-2020):', lanina_years_clim)
print('La Nina DJF years (1981-2006):', lanina_years_past)
print('La Nina DJF years (2007-2020):', lanina_years_recent)

rain_clim_lanina = rain_clim.sel(time=rain_clim.djf_year.isin(lanina_years_clim))
rain_past_lanina = rain_past.sel(time=rain_past.djf_year.isin(lanina_years_past))
rain_recent_lanina = rain_recent.sel(time=rain_recent.djf_year.isin(lanina_years_recent))

rain_clim_lanina, rain_past_lanina, rain_recent_lanina = dask.compute(
    rain_clim_lanina.mean('time'),
    rain_past_lanina.mean('time'),
    rain_recent_lanina.mean('time'),
)

rain_clim_lanina = rain_clim_lanina.transpose('lat', 'lon')
rain_past_lanina = rain_past_lanina.transpose('lat', 'lon')
rain_recent_lanina = rain_recent_lanina.transpose('lat', 'lon')

rain_clim_lanina_anom = rain_clim_lanina - full_plot
rain_past_lanina_anom = rain_past_lanina - past_plot
rain_recent_lanina_anom = rain_recent_lanina - recent_plot

rain_clim_lanina_anom = rain_clim_lanina_anom.reset_coords(drop=True)
rain_past_lanina_anom = rain_past_lanina_anom.reset_coords(drop=True)
rain_recent_lanina_anom = rain_recent_lanina_anom.reset_coords(drop=True)


In [ ]:
#data, title, cmap, levels, ticks, cbar_label, extend
plots = [
    (rain_clim_lanina_anom, '(a) Komposit La Nina DJF (1981 - 2020)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    (rain_past_lanina_anom, '(b) Komposit La Nina DJF (1981 - 2006)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    (rain_recent_lanina_anom, '(c) Komposit La Nina DJF (2007 - 2020)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    ((rain_recent_lanina_anom - rain_past_lanina_anom), '(d) Selisih (c) - (b)', 'BrBG', 'Selisih (mm/bulan)', np.arange(-100, 101, 5), np.arange(-100, 101, 25), 'both'),
]


In [ ]:
for data, title, cmap, cbar_label, levels, ticks, extend in plots:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(ip_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(ip_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(ip_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


In [ ]:
for data, title, cmap, cbar_label, levels, ticks, extend in plots:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


# 5. Plot Correlation

Compute the DJF teleconnection as Pearson correlation between DJF-mean rainfall and DJF-mean Niño3.4, then plot the correlation maps (use plotting style as previous cell, plto for indo pacific and for maritime continent).

Requirements:
- Use one DJF seasonal mean rainfall value per djf_year
- Use one DJF seasonal mean Niño3.4 value per djf_year
- Keep the DJF year convention as already defined
- Compute correlation at each grid point for: full period, past period, recent period. 
- Plot the three correlation maps with the same color scale
- Also plot a fourth map: recent minus past correlation
- Keep the existing domain, projection, and plot style as much as possible
- Do not add regression yet
- Print the sample size used for each period.

In [ ]:
# Compute DJF-mean rainfall and DJF-mean Niño3.4, then correlate by djf_year
rain_clim_djf_corr = rain_clim.groupby('djf_year').mean('time')
rain_past_djf_corr = rain_past.groupby('djf_year').mean('time')
rain_recent_djf_corr = rain_recent.groupby('djf_year').mean('time')

nino34_clim_djf_corr_series = nino34_clim.groupby('djf_year')[nino34_column].mean()
nino34_past_djf_corr_series = nino34_past.groupby('djf_year')[nino34_column].mean()
nino34_recent_djf_corr_series = nino34_recent.groupby('djf_year')[nino34_column].mean()

nino34_clim_djf_corr = xr.DataArray(
    nino34_clim_djf_corr_series.to_numpy(),
    coords={'djf_year': nino34_clim_djf_corr_series.index.to_numpy()},
    dims='djf_year',
    name='nino34',
)
nino34_past_djf_corr = xr.DataArray(
    nino34_past_djf_corr_series.to_numpy(),
    coords={'djf_year': nino34_past_djf_corr_series.index.to_numpy()},
    dims='djf_year',
    name='nino34',
)
nino34_recent_djf_corr = xr.DataArray(
    nino34_recent_djf_corr_series.to_numpy(),
    coords={'djf_year': nino34_recent_djf_corr_series.index.to_numpy()},
    dims='djf_year',
    name='nino34',
)

rain_clim_djf_corr, nino34_clim_djf_corr = xr.align(rain_clim_djf_corr, nino34_clim_djf_corr, join='inner')
rain_past_djf_corr, nino34_past_djf_corr = xr.align(rain_past_djf_corr, nino34_past_djf_corr, join='inner')
rain_recent_djf_corr, nino34_recent_djf_corr = xr.align(rain_recent_djf_corr, nino34_recent_djf_corr, join='inner')

corr_clim, corr_past, corr_recent = dask.compute(
    xr.corr(rain_clim_djf_corr, nino34_clim_djf_corr, dim='djf_year'),
    xr.corr(rain_past_djf_corr, nino34_past_djf_corr, dim='djf_year'),
    xr.corr(rain_recent_djf_corr, nino34_recent_djf_corr, dim='djf_year'),
)
corr_recent_minus_past = corr_recent - corr_past

print('Sample size full:', int(rain_clim_djf_corr.sizes['djf_year']))
print('Sample size past:', int(rain_past_djf_corr.sizes['djf_year']))
print('Sample size recent:', int(rain_recent_djf_corr.sizes['djf_year']))


In [ ]:
# Student t-test significance for the DJF teleconnection maps
from scipy.stats import t as student_t, norm

def corr_pvalue(r, n):
    r = xr.where(np.abs(r) >= 0.999999, np.sign(r) * 0.999999, r)
    t_stat = r * np.sqrt((n - 2) / (1 - r ** 2))
    return xr.apply_ufunc(
        lambda x: 2 * student_t.sf(np.abs(x), df=n - 2),
        t_stat,
        vectorize=True,
        dask='allowed',
        output_dtypes=[float],
    )

# Fisher's z-test for difference in correlation coefficients between two independent samples
def corr_diff_pvalue(r1, n1, r2, n2):
    r1 = xr.where(np.abs(r1) >= 0.999999, np.sign(r1) * 0.999999, r1)
    r2 = xr.where(np.abs(r2) >= 0.999999, np.sign(r2) * 0.999999, r2)
    z1 = np.arctanh(r1)
    z2 = np.arctanh(r2)
    z_stat = (z2 - z1) / np.sqrt(1 / (n2 - 3) + 1 / (n1 - 3))
    return xr.apply_ufunc(
        lambda x: 2 * norm.sf(np.abs(x)),
        z_stat,
        vectorize=True,
        dask='allowed',
        output_dtypes=[float],
    )

n_full = int(rain_clim_djf_corr.sizes["djf_year"])
n_past = int(rain_past_djf_corr.sizes["djf_year"])
n_recent = int(rain_recent_djf_corr.sizes["djf_year"])

corr_clim_p = corr_pvalue(corr_clim, n_full)
corr_past_p = corr_pvalue(corr_past, n_past)
corr_recent_p = corr_pvalue(corr_recent, n_recent)
corr_recent_minus_past_p = corr_diff_pvalue(corr_past, n_past, corr_recent, n_recent)

corr_clim_sig = corr_clim_p < 0.05
corr_past_sig = corr_past_p < 0.05
corr_recent_sig = corr_recent_p < 0.05
corr_recent_minus_past_sig = corr_recent_minus_past_p < 0.05


In [ ]:

def add_stippling(ax, sig_mask, block=8, size=15, alpha=0.9):
    sig = sig_mask.fillna(False).astype(np.int8)

    # build a much coarser plotting mask
    sig_plot = (sig.coarsen(lat=block, lon=block, boundary='trim').max() > 0)

    yy, xx = np.where(sig_plot.values)
    if yy.size == 0:
        return
    ax.scatter(
        sig_plot['lon'].values[xx],
        sig_plot['lat'].values[yy],
        s=size,
        c='k',
        marker='.',
        linewidths=0,
        alpha=alpha,
        transform=ccrs.PlateCarree(),
        zorder=4,
    )

for name, sig in [('full', corr_clim_sig), ('past', corr_past_sig), ('recent', corr_recent_sig), ('diff', corr_recent_minus_past_sig)]:
    frac = float(sig.fillna(False).astype(int).mean().compute()) * 100
    print(f'{name}: {frac:.1f}% significant at native grid')


In [ ]:
#plot indo pasifik

corr_levels = np.arange(-1, 1.01, 0.05)
corr_ticks = np.arange(-1, 1.01, 0.25)
diff_levels = np.arange(-1, 1.01, 0.05)
diff_ticks = np.arange(-1, 1.01, 0.25)

plot_defs = [
    (corr_clim, corr_clim_sig, '(a) Korelasi Curah Hujan vs Nino3.4 DJF 1981-2020', 'bwr', 'Koefisien korelasi', corr_levels, corr_ticks, 'both'),
    (corr_past, corr_past_sig, '(b) Korelasi Curah Hujan vs Nino3.4 DJF 1981-2006', 'bwr', 'Koefisien korelasi', corr_levels, corr_ticks, 'both'),
    (corr_recent, corr_recent_sig, '(c) Korelasi Curah Hujan vs Nino3.4 DJF 2007-2020', 'bwr', 'Koefisien korelasi', corr_levels, corr_ticks, 'both'),
    (corr_recent_minus_past, corr_recent_minus_past_sig, '(d) Selisih (c) - (b)', 'bwr', 'Selisih korelasi', diff_levels, diff_ticks, 'both'),
]

for data, sig_mask, title, cmap, cbar_label, levels, ticks, extend in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    add_stippling(ax, sig_mask, block=25, size=15, alpha=0.7)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(ip_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(ip_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(ip_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


In [ ]:
# plot maritime continent

corr_levels = np.arange(-1, 1.01, 0.05)
corr_ticks = np.arange(-1, 1.01, 0.25)
diff_levels = np.arange(-1, 1.01, 0.05)
diff_ticks = np.arange(-1, 1.01, 0.25)

plot_defs = [
    (corr_clim, corr_clim_sig, '(a) Korelasi Curah Hujan vs Nino3.4 DJF 1981-2020', 'bwr', 'Koefisien korelasi', corr_levels, corr_ticks, 'both'),
    (corr_past, corr_past_sig, '(b) Korelasi Curah Hujan vs Nino3.4 DJF 1981-2006', 'bwr', 'Koefisien korelasi', corr_levels, corr_ticks, 'both'),
    (corr_recent, corr_recent_sig, '(c) Korelasi Curah Hujan vs Nino3.4 DJF 2007-2020', 'bwr', 'Koefisien korelasi', corr_levels, corr_ticks, 'both'),
    (corr_recent_minus_past, corr_recent_minus_past_sig, '(d) Selisih (c) - (b)', 'bwr', 'Selisih korelasi', diff_levels, diff_ticks, 'both'),
]

for data, sig_mask, title, cmap, cbar_label, levels, ticks, extend in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    add_stippling(ax, sig_mask, block=8, size=15, alpha=0.7)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


# 6. Plot Regresi

In [ ]:
# Reuse the DJF-mean rainfall and DJF-mean Niño3.4 from the correlation section, then regress rainfall on Niño3.4 by djf_year
from scipy.stats import linregress

def _linregress_1d(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan, np.nan, np.nan, np.nan, np.nan
    result = linregress(x[mask], y[mask])
    return result.slope, result.intercept, result.rvalue, result.pvalue, result.stderr

rain_clim_djf_reg, nino34_clim_djf_reg = xr.align(rain_clim_djf_corr, nino34_clim_djf_corr, join='inner')
rain_past_djf_reg, nino34_past_djf_reg = xr.align(rain_past_djf_corr, nino34_past_djf_corr, join='inner')
rain_recent_djf_reg, nino34_recent_djf_reg = xr.align(rain_recent_djf_corr, nino34_recent_djf_corr, join='inner')

rain_clim_djf_reg = rain_clim_djf_reg.chunk({'djf_year': -1})
nino34_clim_djf_reg = nino34_clim_djf_reg.chunk({'djf_year': -1})
rain_past_djf_reg = rain_past_djf_reg.chunk({'djf_year': -1})
nino34_past_djf_reg = nino34_past_djf_reg.chunk({'djf_year': -1})
rain_recent_djf_reg = rain_recent_djf_reg.chunk({'djf_year': -1})
nino34_recent_djf_reg = nino34_recent_djf_reg.chunk({'djf_year': -1})

reg_clim = xr.apply_ufunc(
    _linregress_1d,
    nino34_clim_djf_reg,
    rain_clim_djf_reg,
    input_core_dims=[['djf_year'], ['djf_year']],
    output_core_dims=[[], [], [], [], []],
    vectorize=True,
    dask='parallelized',
    output_dtypes=[float, float, float, float, float],
)
reg_past = xr.apply_ufunc(
    _linregress_1d,
    nino34_past_djf_reg,
    rain_past_djf_reg,
    input_core_dims=[['djf_year'], ['djf_year']],
    output_core_dims=[[], [], [], [], []],
    vectorize=True,
    dask='parallelized',
    output_dtypes=[float, float, float, float, float],
)
reg_recent = xr.apply_ufunc(
    _linregress_1d,
    nino34_recent_djf_reg,
    rain_recent_djf_reg,
    input_core_dims=[['djf_year'], ['djf_year']],
    output_core_dims=[[], [], [], [], []],
    vectorize=True,
    dask='parallelized',
    output_dtypes=[float, float, float, float, float],
)

reg_clim_slope, reg_clim_intercept, reg_clim_r, reg_clim_p, reg_clim_stderr = dask.compute(*reg_clim)
reg_past_slope, reg_past_intercept, reg_past_r, reg_past_p, reg_past_stderr = dask.compute(*reg_past)
reg_recent_slope, reg_recent_intercept, reg_recent_r, reg_recent_p, reg_recent_stderr = dask.compute(*reg_recent)
reg_recent_minus_past_slope = reg_recent_slope - reg_past_slope

reg_clim_sig = reg_clim_p < 0.05
reg_past_sig = reg_past_p < 0.05
reg_recent_sig = reg_recent_p < 0.05

reg_absmax = xr.concat([
    np.abs(reg_clim_slope),
    np.abs(reg_past_slope),
    np.abs(reg_recent_slope),
    np.abs(reg_recent_minus_past_slope),
], dim='stack').max()
reg_absmax = float(reg_absmax)
reg_absmax = max(reg_absmax, 0.05)
reg_absmax = np.ceil(reg_absmax / 0.05) * 0.05
reg_levels = np.linspace(-reg_absmax, reg_absmax, 21)
reg_ticks = np.linspace(-reg_absmax, reg_absmax, 9)

print('Sample size full:', int(rain_clim_djf_reg.sizes['djf_year']))
print('Sample size past:', int(rain_past_djf_reg.sizes['djf_year']))
print('Sample size recent:', int(rain_recent_djf_reg.sizes['djf_year']))


In [ ]:
reg_levels = np.arange(-200, 201, 10)
reg_ticks = np.arange(-200, 201, 50)

In [ ]:
# plot regression indo pacific

plot_defs = [
    (reg_clim_slope, reg_clim_sig, '(a) Regresi Curah Hujan vs Nino3.4 DJF 1981-2020', 'bwr_r', 'Kemiringan regresi curah hujan (mm/hari per 1°C anomali Niño3.4)', reg_levels, reg_ticks, 'both'),
    (reg_past_slope, reg_past_sig, '(b) Regresi Curah Hujan vs Nino3.4 DJF 1981-2006', 'bwr_r', 'Kemiringan regresi curah hujan (mm/hari per 1°C anomali Niño3.4)', reg_levels, reg_ticks, 'both'),
    (reg_recent_slope, reg_recent_sig, '(c) Regresi Curah Hujan vs Nino3.4 DJF 2007-2020', 'bwr_r', 'Kemiringan regresi curah hujan (mm/hari per 1°C anomali Niño3.4)', reg_levels, reg_ticks, 'both'),
    (reg_recent_minus_past_slope, None, '(d) Selisih kemiringan (c) - (b)', 'bwr_r', 'Selisih kemiringan regresi curah hujan (mm/hari per 1°C anomali Niño3.4)', reg_levels, reg_ticks, 'both'),
]

for data, sig_mask, title, cmap, cbar_label, levels, ticks, extend in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    if sig_mask is not None:
        add_stippling(ax, sig_mask, block=25, size=15, alpha=0.9)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(ip_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(ip_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(ip_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


In [ ]:
# plot regression maritime continent

plot_defs = [
    (reg_clim_slope, reg_clim_sig, '(a) Regresi Curah Hujan vs Nino3.4 DJF 1981-2020', 'bwr', 'Kemiringan regresi curah hujan (mm/hari per 1°C anomali Niño3.4)', reg_levels, reg_ticks, 'both'),
    (reg_past_slope, reg_past_sig, '(b) Regresi Curah Hujan vs Nino3.4 DJF 1981-2006', 'bwr', 'Kemiringan regresi curah hujan (mm/hari per 1°C anomali Niño3.4)', reg_levels, reg_ticks, 'both'),
    (reg_recent_slope, reg_recent_sig, '(c) Regresi Curah Hujan vs Nino3.4 DJF 2007-2020', 'bwr', 'Kemiringan regresi curah hujan (mm/hari per 1°C anomali Niño3.4)', reg_levels, reg_ticks, 'both'),
    (reg_recent_minus_past_slope, None, '(d) Selisih kemiringan (c) - (b)', 'bwr', 'Selisih kemiringan regresi curah hujan (mm/hari per 1°C anomali Niño3.4)', reg_levels, reg_ticks, 'both'),
]

for data, sig_mask, title, cmap, cbar_label, levels, ticks, extend in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    if sig_mask is not None:
        add_stippling(ax, sig_mask, block=8, size=15, alpha=0.9)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


In [ ]:
#data, title, cmap, levels, ticks, cbar_label, extend
plots = [
    (rain_clim_lanina_anom, '(a) Komposit La Nina DJF (1981 - 2020)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    (rain_past_lanina_anom, '(b) Komposit La Nina DJF (1981 - 2006)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    (rain_recent_lanina_anom, '(c) Komposit La Nina DJF (2007 - 2020)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    ((rain_recent_lanina_anom - rain_past_lanina_anom), '(d) Selisih (c) - (b)', 'BrBG', 'Selisih (mm/bulan)', np.arange(-100, 101, 5), np.arange(-100, 101, 25), 'both'),
]
